# 07 Real Pseudo-3D Sleipner Review

## Context
This journal entry reviews the first real pseudo-3D public Sleipner benchmark built from 11 neighboring inlines. The important outcome is now clear: the original hybrid-centered pseudo-3D idea did not become the best field method, but the benchmark itself was useful because it revealed that `plain_ml_structured_constrained` is the strongest current field-facing support-mapping result.


## What Was Added
- A real 11-inline public benchmark for the p07 temporal sequence (`1994 -> 2001, 2004, 2006`).
- A real 11-inline public benchmark for the p10 direct benchmark (`1994 -> 2010`).
- A pseudo-3D consistency stage over inline neighborhoods.
- A layer-aware structured support variant tested against the same benchmark.
- A stronger matched-warp classical comparator tested under the same benchmark constraints.
- Volume-level field metrics: support IoU, support-volume IoU, coverage, outside-support fraction, crossline continuity, connected components, and compactness.
- Multi-inline `zarr` volumes and 3D/4D visual outputs.


In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path('/Users/ameerfiras/Propagation')
P07_VOLUME = ROOT / 'runs' / 'sleipner_p07_11inline' / 'results' / 'volume_manifest.json'
P10_VOLUME = ROOT / 'runs' / 'sleipner_p10_11inline' / 'results' / 'volume_manifest.json'

def load_json(path):
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)

def volume_rows(payload):
    volume_summary = payload.get('volume_summary', {})
    rows = []
    for vintage, methods in volume_summary.get('by_vintage', {}).items():
        for method, metrics in methods.items():
            for metric, value in metrics.items():
                rows.append({'scope': 'by_vintage', 'vintage': vintage, 'method': method, 'metric': metric, 'value': value})
    for method, metrics in volume_summary.get('overall', {}).items():
        for metric, value in metrics.items():
            rows.append({'scope': 'overall', 'vintage': 'all', 'method': method, 'metric': metric, 'value': value})
    return pd.DataFrame(rows)

p07_volume = volume_rows(load_json(P07_VOLUME)) if P07_VOLUME.exists() else pd.DataFrame()
p10_volume = volume_rows(load_json(P10_VOLUME)) if P10_VOLUME.exists() else pd.DataFrame()
P07_VOLUME.exists(), P10_VOLUME.exists()

## p07 Temporal Volume Review
This section checks whether the pseudo-3D pipeline produces coherent support growth and stable crossline behavior across the 2001, 2004, and 2006 monitors.


In [ ]:
if p07_volume.empty:
    print('p07 volume metrics not generated yet')
else:
    display(
        p07_volume[
            p07_volume['method'].isin([
                'best_classical_constrained',
                'plain_ml_constrained',
                'plain_ml_structured_constrained',
                'plain_ml_layered_structured_constrained',
                'hybrid_ml_constrained',
            ])
        ].pivot_table(index=['vintage', 'metric'], columns='method', values='value', aggfunc='first')
    )


## p10 Direct Volume Review
This section checks the stricter same-year benchmark: whether the pseudo-3D stage improves 2010 support alignment and support-volume occupancy across the full 11-inline block.


In [ ]:
if p10_volume.empty:
    print('p10 volume metrics not generated yet')
else:
    display(
        p10_volume[
            p10_volume['method'].isin([
                'best_classical_constrained',
                'plain_ml_constrained',
                'plain_ml_structured_constrained',
                'plain_ml_layered_structured_constrained',
                'hybrid_ml_constrained',
            ])
        ].pivot_table(index=['vintage', 'metric'], columns='method', values='value', aggfunc='first')
    )


## Crossline Coherence Check
The key pseudo-3D question is whether the predicted support becomes more spatially coherent across neighboring inlines instead of flickering section-by-section.


In [ ]:
def continuity_table(frame):
    if frame.empty:
        return frame
    subset = frame[frame['metric'].isin(['crossline_continuity', 'support_volume_iou_2010', 'trace_iou_with_2010_support'])]
    return subset.pivot_table(index=['vintage', 'metric'], columns='method', values='value', aggfunc='first')

display(continuity_table(p07_volume))
display(continuity_table(p10_volume))


## What Looks Useful / What Still Fails
Use this section to keep the project honest.
- The useful result is not the original hybrid pseudo-3D idea. The useful result is that `plain_ml_structured_constrained` improves support-volume IoU while preserving strong lateral alignment.
- The new `plain_ml_layered_structured_constrained` variant was worth testing, but it did not beat `plain_ml_structured_constrained` on the real 11-inline benchmarks.
- A stronger matched-warp classical comparator was also worth testing, but it did not change the direct `p10` headline: `plain_ml_structured_constrained` still beats the best constrained classical baseline there.
- On `p07`, the best constrained classical baseline can still score higher on raw support-volume IoU by staying very broad, but it remains far less selective and much worse on lateral support alignment than `plain_ml_structured_constrained`.
- The older hybrid-centered story is weakened because the stricter pseudo-3D benchmark showed the hybrid field outputs were not the best ones.
- That does not mean the project lost its novelty. It means the novelty moved from `hybrid field method wins` to `public pseudo-3D benchmark plus structured support mapping on real Sleipner data`.
- The benchmark itself is valuable because it caught a weaker method story before it became paper language.


In [ ]:
visual_paths = [
    ROOT / 'runs' / 'sleipner_review_browser.html',
    ROOT / 'runs' / 'sleipner_p07_11inline' / 'results' / 'visualization' / 'slice_browser.html',
    ROOT / 'runs' / 'sleipner_p10_11inline' / 'results' / 'visualization' / 'slice_browser.html',
    ROOT / 'runs' / 'paper_evidence' / 'results' / 'figures' / 'paper_direct_2010_panel.png',
]
for path in visual_paths:
    print(path, 'exists=' + str(path.exists()))


## Next Method Step
The next step is not adding more pseudo-3D variants for their own sake.
The current method of record stays `plain_ml_structured_constrained`, because it beat the layered extension on the real 11-inline benchmarks.
The stronger classical comparator also did not displace it on the direct `p10` benchmark, so the next method change should focus on strengthening the structured plain method itself rather than inventing another adjacent baseline.
The next useful change has to improve the fixed 11-inline benchmark without losing support-volume IoU or crossline continuity.
If a later step changes the science in a meaningful way, then it deserves the next notebook entry.

Future entries should continue the numbered journal series as `08_...`, `09_...`, and so on.
